In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# 1. Load operational datasets
orders = pd.read_csv('../data/olist_orders_dataset.csv')
order_items = pd.read_csv('../data/olist_order_items_dataset.csv')
reviews = pd.read_csv('../data/olist_order_reviews_dataset.csv')

# 2. Parse delivery timestamps to calculate operational delays
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])
orders['order_estimated_delivery_date'] = pd.to_datetime(orders['order_estimated_delivery_date'])

# Calculate actual delivery time (days) and variance against the estimate
orders['actual_delivery_days'] = (orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']).dt.days
orders['delivery_vs_estimated'] = (orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']).dt.days

# 3. Aggregate items pricing/freight per order
items_agg = order_items.groupby('order_id').agg({
    'price': 'sum',
    'freight_value': 'sum',
    'product_id': 'count'
}).rename(columns={'product_id': 'total_items'}).reset_index()

# 4. Consolidate into a unified analytical dataframe
ml_df = pd.merge(orders, items_agg, on='order_id')
ml_df = pd.merge(ml_df, reviews[['order_id', 'review_score']], on='order_id')

# Clean missing values resulting from unfulfilled orders
ml_df = ml_df.dropna(subset=['actual_delivery_days', 'review_score'])

print(f"✅ Data processing complete. Feature matrix initialized with {ml_df.shape[0]} samples.")

✅ Data processing complete. Feature matrix initialized with 96359 samples.


In [3]:
# Create binary target variable: 1 = Low Score (1-3 stars), 0 = High Score (4-5 stars)
ml_df['is_low_score'] = np.where(ml_df['review_score'] <= 3, 1, 0)

# Select predictive features
features = ['price', 'freight_value', 'total_items', 'actual_delivery_days', 'delivery_vs_estimated']
X = ml_df[features]
y = ml_df['is_low_score']

# Stratified split to preserve class ratios between train and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print("Class Distribution in Training Set:")
print(y_train.value_counts(normalize=True))

# Initialize Random Forest with balanced class weights to force the model to look at minor classes
model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
model.fit(X_train, y_train)

print("\n🚀 Machine Learning Model trained successfully!")

Class Distribution in Training Set:
is_low_score
0    0.789199
1    0.210801
Name: proportion, dtype: float64

🚀 Machine Learning Model trained successfully!


In [5]:
# Predict evaluation subset
y_pred = model.predict(X_test)

print("--- CLASSIFICATION PERFORMANCE REPORT ---")
print(classification_report(y_test, y_pred))

# Feature Importance evaluation
importances = model.feature_importances_
for feature, importance in zip(features, importances):
    print(f"Feature: {feature:<22} Importance: {importance:.4f}")

--- CLASSIFICATION PERFORMANCE REPORT ---
              precision    recall  f1-score   support

           0       0.83      0.95      0.88     15210
           1       0.57      0.25      0.35      4062

    accuracy                           0.80     19272
   macro avg       0.70      0.60      0.62     19272
weighted avg       0.77      0.80      0.77     19272

Feature: price                  Importance: 0.3138
Feature: freight_value          Importance: 0.2877
Feature: total_items            Importance: 0.0267
Feature: actual_delivery_days   Importance: 0.1459
Feature: delivery_vs_estimated  Importance: 0.2258
